In [0]:
loan_info=spark.read.format("delta").load("abfss://silver@deassessment.dfs.core.windows.net/loans/")
loan_info.createOrReplaceTempView("loan_info")


In [0]:
%sql
select * from loan_info

In [0]:
payment_transaction=spark.read.format("delta").load("abfss://silver@deassessment.dfs.core.windows.net/payment/")
payment_transaction.createOrReplaceTempView("payment_transaction")

In [0]:
%sql
select * from payment_transaction

In [0]:
df_transaction=spark.sql(f"""with extracted_components as (
    select 
        pt.amount as amount,
        pt.loan_id as loan_id,
        pt.metadata as metadata,
        pt.payment_id as payment_id,
        pt.payment_method as payment_method,
        pt.timestamp as timestamp,
        li.origination_date as origination_date,
        li.customer_id as customer_id,
        li.product_type as product_type,
        li.principal_amount as principal_amount,
        EXTRACT(YEAR FROM CAST(pt.timestamp AS TIMESTAMP)) AS target_year,
        EXTRACT(MONTH FROM CAST(pt.timestamp AS TIMESTAMP)) AS target_month,
        EXTRACT(DAY FROM CAST(li.origination_date AS DATE)) AS orig_day
    from payment_transaction pt
    inner join loan_info li on pt.loan_id = li.loan_id
    where li.status in ('active', 'default')
),

rollover_logic as (
    select
        amount,
        loan_id,
        metadata,
        payment_id,
        payment_method,
        timestamp,
        origination_date,
        customer_id,
        product_type,
        target_year,
        target_month,
        orig_day,
        principal_amount,
        -- Apply the calendar logic explicitly 
        CASE 
            WHEN target_month = 2 AND orig_day > 28 THEN 1
            WHEN target_month IN (4, 6, 9, 11) AND orig_day = 31 THEN 1
            ELSE orig_day 
        END AS final_day,
        
        CASE 
            WHEN target_month = 2 AND orig_day > 28 THEN 3 
            WHEN target_month IN (4, 6, 9, 11) AND orig_day = 31 THEN target_month + 1 
            ELSE target_month
        END AS final_month
    from extracted_components
)

select 
    amount,
    loan_id,
    metadata,
    payment_id,
    payment_method,
    timestamp,
    customer_id,
    product_type,
    origination_date,
    principal_amount,
    make_date(target_year, final_month, final_day) AS due_date 
from rollover_logic""")

df_transaction.createOrReplaceTempView("df_transaction")

In [0]:
df_transaction.display()
df_transaction.printSchema()

In [0]:
from pyspark.sql.functions import col, when, datediff, to_date, count, max, sum, stddev, round, lit

# 1. Parse your existing timestamp and due_date strings into clean dates
df_dates = df_transaction.withColumn("due_date_parsed", col("due_date").cast("date")) \
             .withColumn("payment_date_parsed", to_date(col("timestamp")))

# 2. Calculate Days Past Due (DPD) for every historical row
df_with_dpd = df_dates.withColumn(
    "dpd",
    when(col("payment_date_parsed").isNull(), datediff(lit("2026-06-24"), col("due_date_parsed"))) \
    .when(col("payment_date_parsed") > col("due_date_parsed"), datediff(col("payment_date_parsed"), col("due_date_parsed"))) \
    .otherwise(lit(0))
)

# 3. Profile customer habits using only behavioral aggregates
df_behavior_profile = df_with_dpd.groupBy("customer_id", "loan_id").agg(
    count("payment_id").alias("total_payments_made"),
    
    # Timeline Consistency Metrics
    max("dpd").alias("max_days_late_ever"),
    sum(when(col("dpd") > 0, 1).otherwise(0)).alias("number_of_late_payments"),
    
    # Amount Consistency Metrics (Checking if payment sizes change month-to-month)
    round(stddev("amount"), 2).alias("payment_amount_variance")
)

# 4. Filter for customers breaking either timeline or amount consistency rules
# If variance is null (meaning they only made 1 payment so far), we treat it as 0 variance.
df_inconsistent_behavior = df_behavior_profile.filter(
    (col("max_days_late_ever") > 5) |                       # Rule 1: Paid > 5 days late
    (col("number_of_late_payments") >= 2) |                 # Rule 2: Chronic late payer
    (when(col("payment_amount_variance").isNull(), 0).otherwise(col("payment_amount_variance")) > 10.00) # Rule 3: Changes payment amount by more than $10
)

# Show final exception report ordered by the highest timeline breach
df_inconsistent_behavior.orderBy(col("max_days_late_ever").desc()).display()

In [0]:
df_inconsistent_behavior.write.mode('overwrite').parquet("abfss://gold@deassessment.dfs.core.windows.net/inconsistent_payment")